In [ ]:
job_id = str(uuid.uuid4())
job_name = 'DM_FACT_MEDICATION_CONSENT_LOAD'
layer_name = 'DATAMART'
start_time = datetime.now()
rows_loaded = 0
rows_failed = 0
run_status = 'SUCCESS'
error_message = None

try:
    merge_result = session.sql(f"""
    MERGE INTO {DB}.{DATAMART}.FACT_DM_MEDICATION_CONSENT tgt
    USING (
        SELECT
            src.MHP_ID,
            dim_med.DM_MEDICATION_CONSENT_DETAILS_ID,
            dim_cus.DM_CUS_CUSTODIES_ID,
            dim_cas.DM_CAS_CASES_ID,
            src.PERSON_PATIENT_ID AS DM_PER_PERSONS_ID,
            src.MED_MED_ID,
            src.MHP_REPORT_DATE,
            src.MEDICATION_START_DATE,
            src.MEDICATION_END_DATE,
            src.THRIVE_FAILURE_IND,
            src.PERSON_PRESCRIBER_ID,
            src.MHB_ADD_USER,
            src.MHB_ADD_TS,
            src.MHB_ADD_PERSON_ID,
            src.MHB_ADD_ORGN_ID,
            src.MHB_MOD_USER,
            src.MHB_MOD_TS,
            src.MHB_MOD_PERSON_ID,
            src.MHB_MOD_ORGN_ID,
            src.DATE_UNKNOWN,
            src.PRESCRIBER_PCS_PCS_ID,
            src.LAST_REFILL_DATE,
            src.CONSENT_DATE,
            src.CONSENT_BY_PERSON_ID,
            src.MED_ID,
            src.ROGER_DECISION_DRUG_IND,
            src.MED_ADD_TS,
            src.MED_ADD_USER,
            src.MED_MOD_TS,
            src.MED_MOD_USER,
            src.ALLOW_FREQUENCY_NUM,
            src.UNIT_ORGN_ID,
            src.AREA_ORGN_ID,
            src.REGION_ORGN_ID,
            src.CASE_WORKER_ID,
            SHA2(CONCAT_WS('|',
                COALESCE(TO_VARCHAR(src.MHP_ID), ''),
                COALESCE(TO_VARCHAR(src.MED_MED_ID), ''),
                COALESCE(TO_VARCHAR(src.MHP_REPORT_DATE), ''),
                COALESCE(TO_VARCHAR(src.MEDICATION_START_DATE), ''),
                COALESCE(TO_VARCHAR(src.MEDICATION_END_DATE), ''),
                COALESCE(src.THRIVE_FAILURE_IND, ''),
                COALESCE(TO_VARCHAR(src.PERSON_PRESCRIBER_ID), ''),
                COALESCE(TO_VARCHAR(src.PERSON_PATIENT_ID), ''),
                COALESCE(src.MHB_ADD_USER, ''),
                COALESCE(TO_VARCHAR(src.MHB_ADD_TS), ''),
                COALESCE(TO_VARCHAR(src.MHB_ADD_PERSON_ID), ''),
                COALESCE(TO_VARCHAR(src.MHB_ADD_ORGN_ID), ''),
                COALESCE(src.MHB_MOD_USER, ''),
                COALESCE(TO_VARCHAR(src.MHB_MOD_TS), ''),
                COALESCE(TO_VARCHAR(src.MHB_MOD_PERSON_ID), ''),
                COALESCE(TO_VARCHAR(src.MHB_MOD_ORGN_ID), ''),
                COALESCE(src.DATE_UNKNOWN, ''),
                COALESCE(TO_VARCHAR(src.PRESCRIBER_PCS_PCS_ID), ''),
                COALESCE(TO_VARCHAR(src.LAST_REFILL_DATE), ''),
                COALESCE(TO_VARCHAR(src.CONSENT_DATE), ''),
                COALESCE(TO_VARCHAR(src.CONSENT_BY_PERSON_ID), ''),
                COALESCE(TO_VARCHAR(src.MED_ID), ''),
                COALESCE(src.ROGER_DECISION_DRUG_IND, ''),
                COALESCE(TO_VARCHAR(src.MED_ADD_TS), ''),
                COALESCE(src.MED_ADD_USER, ''),
                COALESCE(TO_VARCHAR(src.MED_MOD_TS), ''),
                COALESCE(src.MED_MOD_USER, ''),
                COALESCE(TO_VARCHAR(src.ALLOW_FREQUENCY_NUM), ''),
                COALESCE(TO_VARCHAR(src.UNIT_ORGN_ID), ''),
                COALESCE(TO_VARCHAR(src.AREA_ORGN_ID), ''),
                COALESCE(TO_VARCHAR(src.REGION_ORGN_ID), ''),
                COALESCE(TO_VARCHAR(src.CAS_ID), ''),
                COALESCE(TO_VARCHAR(src.CUS_ID), ''),
                COALESCE(TO_VARCHAR(src.CASE_WORKER_ID), '')
            ), 256) AS CHECKSUM
        FROM {DB}.{DATAMART}.{DM_SOURCE} src
        LEFT JOIN {DB}.{DATAMART}.DIM_DM_MEDICATION_CONSENT_DETAILS_DT dim_med
            ON dim_med.DM_MEDICATION_CONSENT_DETAILS_ID = src.MHP_ID
            AND dim_med.UPDATED_DATE IS NULL
        LEFT JOIN {DB}.{DATAMART}.DIM_DM_CUS_CUSTODIES_DT dim_cus
            ON dim_cus.CUS_ID = src.CUS_ID
            AND dim_cus.UPDATED_DATE IS NULL
        LEFT JOIN {DB}.{DATAMART}.DIM_DM_CAS_CASES_DT dim_cas
            ON dim_cas.CAS_ID = src.CAS_ID
            AND dim_cas.UPDATED_DATE IS NULL
        WHERE src.RN_POI = 1 AND src.RN_CA = 1
    ) src
    ON tgt.MHP_ID = src.MHP_ID
    WHEN MATCHED AND tgt.CHECKSUM <> src.CHECKSUM THEN UPDATE SET
        DM_MEDICATION_CONSENT_DETAILS_ID = src.DM_MEDICATION_CONSENT_DETAILS_ID,
        DM_CUS_CUSTODIES_ID = src.DM_CUS_CUSTODIES_ID,
        DM_CAS_CASES_ID = src.DM_CAS_CASES_ID,
        DM_PER_PERSONS_ID = src.DM_PER_PERSONS_ID,
        MED_MED_ID = src.MED_MED_ID,
        MHP_REPORT_DATE = src.MHP_REPORT_DATE,
        MEDICATION_START_DATE = src.MEDICATION_START_DATE,
        MEDICATION_END_DATE = src.MEDICATION_END_DATE,
        THRIVE_FAILURE_IND = src.THRIVE_FAILURE_IND,
        PERSON_PRESCRIBER_ID = src.PERSON_PRESCRIBER_ID,
        MHB_ADD_USER = src.MHB_ADD_USER,
        MHB_ADD_TS = src.MHB_ADD_TS,
        MHB_ADD_PERSON_ID = src.MHB_ADD_PERSON_ID,
        MHB_ADD_ORGN_ID = src.MHB_ADD_ORGN_ID,
        MHB_MOD_USER = src.MHB_MOD_USER,
        MHB_MOD_TS = src.MHB_MOD_TS,
        MHB_MOD_PERSON_ID = src.MHB_MOD_PERSON_ID,
        MHB_MOD_ORGN_ID = src.MHB_MOD_ORGN_ID,
        DATE_UNKNOWN = src.DATE_UNKNOWN,
        PRESCRIBER_PCS_PCS_ID = src.PRESCRIBER_PCS_PCS_ID,
        LAST_REFILL_DATE = src.LAST_REFILL_DATE,
        CONSENT_DATE = src.CONSENT_DATE,
        CONSENT_BY_PERSON_ID = src.CONSENT_BY_PERSON_ID,
        MED_ID = src.MED_ID,
        ROGER_DECISION_DRUG_IND = src.ROGER_DECISION_DRUG_IND,
        MED_ADD_TS = src.MED_ADD_TS,
        MED_ADD_USER = src.MED_ADD_USER,
        MED_MOD_TS = src.MED_MOD_TS,
        MED_MOD_USER = src.MED_MOD_USER,
        ALLOW_FREQUENCY_NUM = src.ALLOW_FREQUENCY_NUM,
        UNIT_ORGN_ID = src.UNIT_ORGN_ID,
        AREA_ORGN_ID = src.AREA_ORGN_ID,
        REGION_ORGN_ID = src.REGION_ORGN_ID,
        CASE_WORKER_ID = src.CASE_WORKER_ID,
        CHECKSUM = src.CHECKSUM
    WHEN NOT MATCHED THEN INSERT (
        MHP_ID, DM_MEDICATION_CONSENT_DETAILS_ID, DM_CUS_CUSTODIES_ID, DM_CAS_CASES_ID,
        DM_PER_PERSONS_ID, MED_MED_ID, MHP_REPORT_DATE, MEDICATION_START_DATE, MEDICATION_END_DATE,
        THRIVE_FAILURE_IND, PERSON_PRESCRIBER_ID, MHB_ADD_USER, MHB_ADD_TS,
        MHB_ADD_PERSON_ID, MHB_ADD_ORGN_ID, MHB_MOD_USER, MHB_MOD_TS,
        MHB_MOD_PERSON_ID, MHB_MOD_ORGN_ID, DATE_UNKNOWN, PRESCRIBER_PCS_PCS_ID,
        LAST_REFILL_DATE, CONSENT_DATE, CONSENT_BY_PERSON_ID, MED_ID,
        ROGER_DECISION_DRUG_IND, MED_ADD_TS, MED_ADD_USER, MED_MOD_TS, MED_MOD_USER,
        ALLOW_FREQUENCY_NUM, UNIT_ORGN_ID, AREA_ORGN_ID, REGION_ORGN_ID,
        CASE_WORKER_ID, CREATED_DATE, CHECKSUM
    ) VALUES (
        src.MHP_ID, src.DM_MEDICATION_CONSENT_DETAILS_ID, src.DM_CUS_CUSTODIES_ID, src.DM_CAS_CASES_ID,
        src.DM_PER_PERSONS_ID, src.MED_MED_ID, src.MHP_REPORT_DATE, src.MEDICATION_START_DATE, src.MEDICATION_END_DATE,
        src.THRIVE_FAILURE_IND, src.PERSON_PRESCRIBER_ID, src.MHB_ADD_USER, src.MHB_ADD_TS,
        src.MHB_ADD_PERSON_ID, src.MHB_ADD_ORGN_ID, src.MHB_MOD_USER, src.MHB_MOD_TS,
        src.MHB_MOD_PERSON_ID, src.MHB_MOD_ORGN_ID, src.DATE_UNKNOWN, src.PRESCRIBER_PCS_PCS_ID,
        src.LAST_REFILL_DATE, src.CONSENT_DATE, src.CONSENT_BY_PERSON_ID, src.MED_ID,
        src.ROGER_DECISION_DRUG_IND, src.MED_ADD_TS, src.MED_ADD_USER, src.MED_MOD_TS, src.MED_MOD_USER,
        src.ALLOW_FREQUENCY_NUM, src.UNIT_ORGN_ID, src.AREA_ORGN_ID, src.REGION_ORGN_ID,
        src.CASE_WORKER_ID, CURRENT_TIMESTAMP(), src.CHECKSUM
    )
    """).collect()

    rows_inserted = merge_result[0][0]
    rows_updated = merge_result[0][1]
    rows_loaded = rows_inserted + rows_updated

    session.call(f"{DB}.{AUDIT}.{SP_LOG_AUDIT}", job_id, job_name, layer_name, start_time, datetime.now(), rows_loaded, rows_failed, run_status, None)
    session.call(f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}", run_status, job_name, layer_name, f'FACT_DM_MEDICATION_CONSENT loaded. Inserted: {rows_inserted}, Updated: {rows_updated}')
    print(f'SUCCESS | Inserted: {rows_inserted} | Updated: {rows_updated}')

except Exception as error:
    run_status = 'FAILED'
    rows_failed = 1
    error_message = str(error)
    session.call(f"{DB}.{AUDIT}.{SP_LOG_AUDIT}", job_id, job_name, layer_name, start_time, datetime.now(), 0, rows_failed, run_status, error_message)
    session.call(f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}", run_status, job_name, layer_name, f'FACT_DM_MEDICATION_CONSENT failed: {error_message}')
    print(f'FAILED: {error_message}')